# Denoise Demo

Import

In [ ]:
from __future__ import annotations

import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pyiqa
import torch
from scipy.stats import skew
from torch import Tensor

from mon import Path, Size, transform as T
from mon.models import IZS_N2N, ZS_N2N
from mon.ops import to_image_array, write_image

Setup environment

In [ ]:
current_dir = Path(os.getcwd())
root_dir = current_dir.parents[0]
data_dir = current_dir / "demo"
output_dir = current_dir / "demo"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = {}

# Debug
print(f"Device: {device}")

Resolve I/O

In [ ]:
# image_name = "993_UHD_LL"
# image_name = "1513_UHD_LL"
# image_name = "a0016-jmac_MG_0795"
# image_name = "lol_v1_22"
# image_name = "lsrw_2038"
image_name = "sice_1"

image_file = (data_dir / image_name / f"{image_name}").image_file
depth_file = (data_dir / image_name / f"{image_name}_depth").image_file
target_file = (data_dir / image_name / f"{image_name}_target").image_file

# Load data
image = cv2.imread(str(image_file))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

depth = cv2.imread(str(depth_file), cv2.IMREAD_GRAYSCALE)
if depth.ndim == 2:
    depth = np.expand_dims(depth, axis=-1)

target = cv2.imread(str(target_file))
target = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)

# Debug
print(f"Image shape : {image.shape}")
print(f"Depth shape : {depth.shape}")
print(f"target shape: {target.shape}")
assert image.shape[0:2] == depth.shape[0:2], "'image' and 'depth' must have the same resolution."

Pre-process

In [ ]:
# Normalize and convert to torch.Tensor
transform = T.Compose([
    T.Resize(512, 512),
    T.NormalizeWithMask(normalization="min_max"),
    T.ToTensorV2(transpose_mask=True),
], additional_targets={"target": "image"})
transformed = transform(image=image, mask=depth, target=target)

image_t = transformed["image"]
depth_t = transformed["mask"]
target_t = transformed["target"]

image_t = image_t.unsqueeze(0).to(device)
depth_t = depth_t.unsqueeze(0).to(device)
target_t = target_t.unsqueeze(0).to(device)

# Debug
print(f"Image shape : {image_t.shape}. Type: {image_t.dtype}. Device: {image_t.device}.")
print(f"Depth shape : {depth_t.shape}. Type: {depth_t.dtype}. Device: {depth_t.device}.")
print(f"target shape: {target_t.shape}. Type: {target_t.dtype}. Device: {target_t.device}.")

Process

In [ ]:
def test_zs_n2n(image_: Tensor, verbose: bool = False):
    global results, device

    zs_n2n = ZS_N2N(fit=True, device=device, verbose=verbose)
    outputs = zs_n2n(image_)

    results["zs_n2n"] = {"restored": outputs["restored"], }

In [ ]:
def test_izs_n2n(image_: Tensor, verbose: bool = False):
    global results, device

    izs_n2n = IZS_N2N(fit=True, device=device, verbose=verbose)
    outputs = izs_n2n(image_)

    results["izs_n2n"] = {"restored": outputs["restored"], }

In [ ]:
test_zs_n2n(image_t)

In [ ]:
test_izs_n2n(image_t)

Measure metrics

In [ ]:
def measure_metrics(key: str = "restored"):
    global image_t, target_t, results, device

    # Create metrics with default settings
    psnr_metric = pyiqa.create_metric("psnr", device=device)
    ssim_metric = pyiqa.create_metric("ssim", device=device)
    ssimc_metric = pyiqa.create_metric("ssimc", device=device)
    lpips_metric = pyiqa.create_metric("lpips", device=device)
    niqe_metric = pyiqa.create_metric("niqe", device=device)
    brisque_metric = pyiqa.create_metric("brisque", device=device)

    # Concatenate results for visualization
    titles = ["image"]
    images = [image_t]
    for k1, v1 in results.items():
        titles.append(k1)
        images.append(v1[key])

    # Measure metrics
    target_ = target_t.to(device)
    for i in range(0, len(images)):
        if isinstance(images[i], Tensor):
            image_ = images[i].to(device)
        else:
            image_ = images[i]

        psnr_value = psnr_metric(image_, target_).item()
        ssim_value = ssim_metric(image_, target_).item()
        ssimc_value = ssimc_metric(image_, target_).item()
        lpips_value = lpips_metric(image_, target_).item()
        niqe_value = niqe_metric(image_).item()
        brisque_value = brisque_metric(image_).item()

        print(
            f"{titles[i]:15s} "
            f"| PSNR: {psnr_value:6.4f} "
            f"| SSIM: {ssim_value:6.4f} "
            f"| SSIM-C: {ssimc_value:6.4f} "
            f"| LPIPS: {lpips_value:6.4f} "
            f"| NIQE: {niqe_value:6.4f} "
            f"| BRISQUE: {brisque_value:6.4f} "
        )

In [ ]:
def analyze_noise_distribution(image: Tensor, target: Tensor | None = None):
    """Analyzes whether the noise in a tensor is zero-mean and symmetrical.
    Expects tensors in shape [B, C, H, W] with values 0.0 to 1.0.
    """
    # 1. Extract the Noise
    if target is not None:
        # We have ground truth: pure residual noise
        noise = (image - target).cpu().numpy().flatten()
    else:
        # No ground truth: We crop a known "flat" area (e.g., top-left 50x50 pixels)
        # You should adjust these crop coordinates to a flat wall/sky in your image
        patch = image[:, :, 0:50, 0:50].cpu().numpy().flatten()
        noise = patch - np.mean(patch)

    # 2. Calculate Statistical Properties
    # A perfect symmetrical distribution has a mean of 0.0 and a skewness of 0.0
    noise_mean = np.mean(noise)
    noise_skew = skew(noise)

    print(f"--- Noise Statistics ---")
    print(f"Mean (Should be ~0.0): {noise_mean:.6f}")
    print(f"Skewness (Should be ~0.0): {noise_skew:.6f}")

    if abs(noise_skew) > 0.5:
        print("Warning: The noise is highly asymmetrical (likely clipped by low-light).")
    elif abs(noise_skew) > 0.1:
        print("Note: The noise is slightly asymmetrical.")
    else:
        print("Pass: The noise is approximately symmetrical.")

    # 3. Plot the Histogram
    plt.figure(figsize=(6, 3))

    # We use 100 bins and limit the range to +/- 0.2 (since image max is 1.0)
    plt.hist(noise, bins=100, range=(-0.2, 0.2), density=True, alpha=0.7, color='blue')

    plt.axvline(x=0, color='red', linestyle='--', label='Zero Mean')
    plt.axvline(x=noise_mean, color='green', linestyle='-', label=f'Actual Mean ({noise_mean:.4f})')

    plt.title(f"Noise Distribution (Skewness: {noise_skew:.3f})")
    plt.xlabel("Pixel Value Difference")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
measure_metrics("restored")

In [ ]:
analyze_noise_distribution(image_t, target_t)
analyze_noise_distribution(results["zs_n2n"]["restored"], target_t)
analyze_noise_distribution(results["izs_n2n"]["restored"], target_t)

Post-process

In [ ]:
def postprocess():
    global image, results

    size0 = Size.from_any(image)

    # Convert outputs to numpy arrays
    for k1, v1 in results.items():
        for k2, v2 in v1.items():
            v2 = to_image_array(v2)

            size1 = Size.from_any(v2)
            if size0 != size1:
                v2 = cv2.resize(v2, size0.wh)

            results[k1][k2] = v2

In [ ]:
postprocess()

Visualize

In [ ]:
# Setup matplotlib
plt.rcParams["figure.autolayout"] = True

In [ ]:
def compare_results(key: str = "restored"):
    global results

    # Concatenate results for visualization
    titles = ["image", "depth", "ref"]
    images = [image, depth, ref]
    for k1, v1 in results.items():
        titles.append(k1)
        images.append(v1[key])

    # Create a grid of images
    fig, axes = plt.subplots(1, len(images), figsize=(15, 5))
    for i, ax in enumerate(axes):
        ax.imshow(images[i])
        ax.set_title(titles[i])
        ax.axis("off")

    # Adjust the layout and show the plot
    plt.tight_layout()  # Adjust layout to prevent title overlap
    plt.show()

In [ ]:
compare_results(key="restored")

Save

In [ ]:
def save():
    global results

    for k1, v1 in results.items():
        for k2, v2 in v1.items():
            save_file = output_dir / f"{image_name}_debug" / f"{image_name}_{k1}_{k2}.jpg"
            write_image(v2, save_file)

In [ ]:
save()